In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision  
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [25]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
])
trainset=CIFAR10(root= "./data", train=True, download=False,transform=transform)
testset=CIFAR10(root= "./data", train=False, download=False,transform=transform)

In [26]:
trainloader= DataLoader(trainset, batch_size=64, shuffle=True)
testloader= DataLoader(testset, batch_size=64)

Build the CNN

In [27]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1= nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2= nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3= nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1) 
        self.pool= nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1= nn.Linear(128*4*4, 512)
        self.relu= nn.ReLU()
        self.fc2= nn.Linear(512, 10)
        
        
    def forward(self,x):
        x=self.relu(self.conv1(x))
        x=self.pool(x)
        x=self.relu(self.conv2(x))
        x=self.pool(x)
        x=self.relu(self.conv3(x))
        x=self.pool(x)
        x=x.view(-1, 128*4*4)
        x=self.relu(self.fc1(x))
        x=self.fc2(x)
        return x

In [28]:
model= CNN()

In [29]:
criterion= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters(), lr=0.001)

Training the CNN

In [30]:
epochs=10
for epoch in range(epochs):
    running_loss=0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels= data
        optimizer.zero_grad()
        outputs= model(inputs)
        loss= criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

    #Validation
    model.eval()
    correct=0
    total=0
    with torch.no_grad():
        for data in testloader:
            images, labels= data
            outputs= model(images)
            _, predicted= torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Epoch [{epoch+1}/{epochs}], Accuracy: {100 * correct / total:.2f}%")

    if i % 100 == 99:
        print(f"[{epoch+1}, {i+1}] loss: {running_loss/100:.3f}")
        running_loss=0.0

Epoch [1/10], Accuracy: 63.06%
Epoch [2/10], Accuracy: 67.51%
Epoch [3/10], Accuracy: 73.11%
Epoch [4/10], Accuracy: 74.66%
Epoch [5/10], Accuracy: 75.17%
Epoch [6/10], Accuracy: 76.01%
Epoch [7/10], Accuracy: 75.19%
Epoch [8/10], Accuracy: 75.01%
Epoch [9/10], Accuracy: 75.26%
Epoch [10/10], Accuracy: 76.10%


Evaluate our Model

In [31]:
correct_labels=0
total_labels=0
model.eval()
with torch.no_grad():
    for data in testloader:
        images, labels= data
        outputs= model(images)
        _, predicted= torch.max(outputs.data, 1)
        total_labels += labels.size(0)
        correct_labels += (predicted == labels).sum().item()

print(f"Accuracy: {100 * correct_labels / total_labels:.2f}%")

Accuracy: 76.10%
